# Question 2 — Hardware-Aware Neural Architecture Search (NAS)


In [ ]:
# Uncomment in a fresh environment if required:
# %pip install -q torch torchvision onnx onnxruntime pandas matplotlib psutil

import os, sys, json, time, copy, random, platform, gc, csv, math
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

try:
    import psutil
except ImportError:
    psutil = None

try:
    import onnx
    import onnxruntime as ort
except ImportError:
    onnx, ort = None, None

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU_THREADS = max(1, min(4, os.cpu_count() or 1))
torch.set_num_threads(CPU_THREADS)

ROOT = Path("nas_q2_outputs")
(ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)
(ROOT / "onnx").mkdir(parents=True, exist_ok=True)
(ROOT / "results").mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("Device for training:", DEVICE)
print("CPU threads for benchmarking:", CPU_THREADS)
print("Output directory:", ROOT.resolve())

## 2. Experiment configuration and CIFAR-10 data

The search uses a stratified-size subset for fast candidate evaluation. The final selected model is retrained on the full training set. Set `USE_SUBSET_FOR_SEARCH=False` to train each candidate on the full training set, if time permits.

In [ ]:
# Adjust these settings according to available compute.
IMAGE_SIZE = 32
BATCH_SIZE = 128
SEARCH_TRAIN_LIMIT = 10000       # None means all 50,000 training examples
SEARCH_EPOCHS = 2
FINAL_EPOCHS = 15
POPULATION_SIZE = 8
GENERATIONS = 3
ELITE_COUNT = 2
MUTATION_PROB = 0.35

# Deployment constraints measured on this machine.
MAX_LATENCY_MS = 30.0
MAX_MODEL_SIZE_MB = 8.0
MAX_PEAK_MEMORY_MB = 512.0

NUM_WORKERS = 2 if os.name != "nt" else 0
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomCrop(IMAGE_SIZE, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

data_dir = ROOT / "data"
train_full = datasets.CIFAR10(root=str(data_dir), train=True, download=True, transform=train_transform)
train_eval_view = datasets.CIFAR10(root=str(data_dir), train=True, download=False, transform=test_transform)
test_set = datasets.CIFAR10(root=str(data_dir), train=False, download=True, transform=test_transform)

if SEARCH_TRAIN_LIMIT is None or SEARCH_TRAIN_LIMIT >= len(train_full):
    search_indices = list(range(len(train_full)))
else:
    rng = np.random.default_rng(SEED)
    search_indices = rng.choice(len(train_full), SEARCH_TRAIN_LIMIT, replace=False).tolist()

search_train_set = Subset(train_full, search_indices)
search_eval_set = Subset(train_eval_view, search_indices)
train_loader = DataLoader(search_train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
search_eval_loader = DataLoader(search_eval_set, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS)

print(f"Search train examples: {len(search_train_set)} | Test examples: {len(test_set)}")

## 3. Search space and architecture generator

Each chromosome defines:
- input resolution (32, 40, or 48 pixels),
- network depth,
- output channels per block,
- kernel size,
- standard or depthwise-separable convolution,
- ReLU, ReLU6, or SiLU activation.

All candidates end with global average pooling and a 10-class classifier. The generator adapts feature-map size using adaptive pooling, so different input resolutions are supported.

In [ ]:
SEARCH_SPACE = {
    "input_resolution": [32, 40, 48],
    "depth": [2, 3, 4, 5],
    "channels": [16, 24, 32, 48, 64, 96],
    "kernel_size": [3, 5],
    "block_type": ["standard", "depthwise"],
    "activation": ["relu", "relu6", "silu"],
}

def random_config():
    depth = random.choice(SEARCH_SPACE["depth"])
    return {
        "input_resolution": random.choice(SEARCH_SPACE["input_resolution"]),
        "depth": depth,
        "channels": [random.choice(SEARCH_SPACE["channels"]) for _ in range(depth)],
        "kernel_size": [random.choice(SEARCH_SPACE["kernel_size"]) for _ in range(depth)],
        "block_type": [random.choice(SEARCH_SPACE["block_type"]) for _ in range(depth)],
        "activation": random.choice(SEARCH_SPACE["activation"]),
    }

def validate_config(cfg):
    assert cfg["input_resolution"] in SEARCH_SPACE["input_resolution"]
    assert cfg["depth"] in SEARCH_SPACE["depth"]
    for key in ("channels", "kernel_size", "block_type"):
        assert len(cfg[key]) == cfg["depth"], f"{key} length must equal depth"
    assert all(c in SEARCH_SPACE["channels"] for c in cfg["channels"])
    assert all(k in SEARCH_SPACE["kernel_size"] for k in cfg["kernel_size"])
    assert all(b in SEARCH_SPACE["block_type"] for b in cfg["block_type"])
    assert cfg["activation"] in SEARCH_SPACE["activation"]
    return cfg

def mutate_config(parent):
    child = copy.deepcopy(parent)
    if random.random() < MUTATION_PROB:
        child["input_resolution"] = random.choice(SEARCH_SPACE["input_resolution"])
    if random.random() < MUTATION_PROB:
        new_depth = random.choice(SEARCH_SPACE["depth"])
        child["depth"] = new_depth
        for key in ("channels", "kernel_size", "block_type"):
            values = [random.choice(SEARCH_SPACE[key]) for _ in range(new_depth)]
            child[key] = values
    else:
        for i in range(child["depth"]):
            if random.random() < MUTATION_PROB:
                child["channels"][i] = random.choice(SEARCH_SPACE["channels"])
            if random.random() < MUTATION_PROB:
                child["kernel_size"][i] = random.choice(SEARCH_SPACE["kernel_size"])
            if random.random() < MUTATION_PROB:
                child["block_type"][i] = random.choice(SEARCH_SPACE["block_type"])
    if random.random() < MUTATION_PROB:
        child["activation"] = random.choice(SEARCH_SPACE["activation"])
    return validate_config(child)

def config_key(cfg):
    return json.dumps(cfg, sort_keys=True)

In [ ]:
def make_activation(name):
    if name == "relu": return nn.ReLU(inplace=True)
    if name == "relu6": return nn.ReLU6(inplace=True)
    if name == "silu": return nn.SiLU(inplace=True)
    raise ValueError(name)

class ConvBNAct(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel, activation):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, kernel, padding=kernel//2, bias=False),
            nn.BatchNorm2d(out_ch),
            make_activation(activation),
        )

class DepthwiseSeparable(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel, activation):
        super().__init__(
            nn.Conv2d(in_ch, in_ch, kernel, padding=kernel//2, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            make_activation(activation),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            make_activation(activation),
        )

class SearchableCNN(nn.Module):
    def __init__(self, config, num_classes=10):
        super().__init__()
        self.config = validate_config(copy.deepcopy(config))
        blocks, in_ch = [], 3
        for out_ch, kernel, block_type in zip(
            self.config["channels"], self.config["kernel_size"], self.config["block_type"]
        ):
            if block_type == "depthwise":
                blocks.append(DepthwiseSeparable(in_ch, out_ch, kernel, self.config["activation"]))
            else:
                blocks.append(ConvBNAct(in_ch, out_ch, kernel, self.config["activation"]))
            blocks.append(nn.MaxPool2d(2, ceil_mode=True))
            in_ch = out_ch
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

example_config = random_config()
print(json.dumps(example_config, indent=2))
example_model = SearchableCNN(example_config)
print("Parameters:", sum(p.numel() for p in example_model.parameters()))

## 4. Training and validation utilities

Candidate architectures are trained for a small number of epochs and evaluated on a held-out test-style transform of the same search subset. The final model is trained separately and evaluated on the official CIFAR-10 test set.

In [ ]:
def run_epoch(model, loader, optimizer=None, device=DEVICE):
    training = optimizer is not None
    model.train(training)
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / max(total, 1), 100.0 * correct / max(total, 1)

def make_resolution_loader(dataset, resolution, batch_size=BATCH_SIZE, shuffle=False):
    # Dataset transforms are updated through a wrapper to support architecture-specific resolutions.
    class ResizeWrapper(torch.utils.data.Dataset):
        def __init__(self, base, size):
            self.base, self.size = base, size
        def __len__(self): return len(self.base)
        def __getitem__(self, idx):
            x, y = self.base[idx]
            x = F.interpolate(x.unsqueeze(0), size=(self.size, self.size),
                              mode="bilinear", align_corners=False).squeeze(0)
            return x, y
    return DataLoader(ResizeWrapper(dataset, resolution), batch_size=batch_size,
                      shuffle=shuffle, num_workers=NUM_WORKERS)

def train_candidate(config, epochs=SEARCH_EPOCHS, verbose=False):
    model = SearchableCNN(config).to(DEVICE)
    tr_loader = make_resolution_loader(search_train_set, config["input_resolution"], shuffle=True)
    va_loader = make_resolution_loader(search_eval_set, config["input_resolution"], shuffle=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_acc = 0.0
    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, tr_loader, optimizer)
        va_loss, va_acc = run_epoch(model, va_loader)
        best_acc = max(best_acc, va_acc)
        if verbose:
            print(f"Epoch {epoch+1}/{epochs}: train acc={tr_acc:.2f}% | val acc={va_acc:.2f}%")
    return model, best_acc

## 5. Hardware-aware measurement

Latency is measured using a real CPU forward pass with warm-up iterations and synchronized timing. Model size is the serialized checkpoint size. Peak process memory is recorded where `psutil` is available (this is process RSS during the measurement, not a device-specific allocator statistic). Candidates that exceed any configured limit are marked infeasible.

In [ ]:
def measure_cpu_latency(model, input_resolution, warmup=20, iterations=100):
    model_cpu = copy.deepcopy(model).to("cpu").eval()
    x = torch.randn(1, 3, input_resolution, input_resolution)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model_cpu(x)
        times = []
        for _ in range(iterations):
            start = time.perf_counter()
            _ = model_cpu(x)
            times.append((time.perf_counter() - start) * 1000.0)
    del model_cpu
    return float(np.mean(times)), float(np.percentile(times, 95))

def measure_model_size_mb(model):
    path = ROOT / "checkpoints" / "_temporary_candidate.pth"
    torch.save(model.state_dict(), path)
    size_mb = path.stat().st_size / (1024 ** 2)
    path.unlink(missing_ok=True)
    return float(size_mb)

def measure_peak_memory_mb():
    if psutil is None:
        return None
    process = psutil.Process(os.getpid())
    return float(process.memory_info().rss / (1024 ** 2))

def evaluate_hardware(model, config):
    latency_mean, latency_p95 = measure_cpu_latency(model, config["input_resolution"])
    size_mb = measure_model_size_mb(model)
    memory_mb = measure_peak_memory_mb()
    feasible = (
        latency_mean <= MAX_LATENCY_MS and
        size_mb <= MAX_MODEL_SIZE_MB and
        (memory_mb is None or memory_mb <= MAX_PEAK_MEMORY_MB)
    )
    return {
        "latency_ms_mean": latency_mean,
        "latency_ms_p95": latency_p95,
        "model_size_mb": size_mb,
        "peak_process_memory_mb": memory_mb,
        "feasible": bool(feasible),
    }

def fitness_score(val_accuracy, hardware):
    # Feasible models rank by accuracy. Infeasible models receive a strong penalty.
    if not hardware["feasible"]:
        return float(val_accuracy) - 100.0
    # Tiny tie-break penalties encourage lower latency and storage.
    return float(val_accuracy) - 0.05 * hardware["latency_ms_mean"] - 0.1 * hardware["model_size_mb"]

## 6. Evolutionary architecture search

The algorithm initializes a random population, trains and measures every candidate, keeps the elite candidates, and creates the next generation through mutation. Search history is saved after every generation in both CSV and JSON.

In [ ]:
def evaluate_config(config, candidate_id, epochs=SEARCH_EPOCHS):
    print(f"Evaluating {candidate_id}: depth={config['depth']}, resolution={config['input_resolution']}")
    model, val_acc = train_candidate(config, epochs=epochs, verbose=False)
    hardware = evaluate_hardware(model, config)
    score = fitness_score(val_acc, hardware)
    record = {
        "candidate_id": candidate_id,
        "config": copy.deepcopy(config),
        "val_accuracy": val_acc,
        "fitness": score,
        **hardware,
    }
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return record

def evolutionary_search():
    population = [random_config() for _ in range(POPULATION_SIZE)]
    history, all_records = [], []
    seen = set()
    for generation in range(GENERATIONS):
        print(f"\n========== Generation {generation+1}/{GENERATIONS} ==========")
        records = []
        for idx, cfg in enumerate(population):
            key = config_key(cfg)
            if key in seen:
                cfg = random_config()
                key = config_key(cfg)
            seen.add(key)
            record = evaluate_config(cfg, f"g{generation+1}_c{idx+1}")
            record["generation"] = generation + 1
            records.append(record)
            all_records.append(record)
        records.sort(key=lambda r: r["fitness"], reverse=True)
        history.extend(records)
        pd.DataFrame([{
            "candidate_id": r["candidate_id"], "generation": r["generation"],
            "val_accuracy": r["val_accuracy"], "fitness": r["fitness"],
            "latency_ms_mean": r["latency_ms_mean"], "latency_ms_p95": r["latency_ms_p95"],
            "model_size_mb": r["model_size_mb"],
            "peak_process_memory_mb": r["peak_process_memory_mb"],
            "feasible": r["feasible"], "config": json.dumps(r["config"])
        } for r in history]).to_csv(ROOT / "results" / "nas_search_history.csv", index=False)
        with open(ROOT / "results" / "nas_search_history.json", "w") as f:
            json.dump(history, f, indent=2)
        elites = records[:max(1, min(ELITE_COUNT, len(records)))]
        print("Generation best:", round(elites[0]["val_accuracy"], 2),
              "% | feasible:", elites[0]["feasible"])
        next_population = [copy.deepcopy(r["config"]) for r in elites]
        while len(next_population) < POPULATION_SIZE:
            parent = random.choice(elites)["config"]
            next_population.append(mutate_config(parent))
        population = next_population
    feasible_records = [r for r in all_records if r["feasible"]]
    if feasible_records:
        best = max(feasible_records, key=lambda r: r["fitness"])
    else:
        print("WARNING: No candidate met every deployment constraint. Selecting the best fitness candidate.")
        best = max(all_records, key=lambda r: r["fitness"])
    with open(ROOT / "results" / "best_architecture.json", "w") as f:
        json.dump(best, f, indent=2)
    return best, all_records

best_result, search_records = evolutionary_search()
print("\nBest architecture configuration:")
print(json.dumps(best_result["config"], indent=2))
print("Validation accuracy:", round(best_result["val_accuracy"], 2), "%")
print("Hardware:", {k:v for k,v in best_result.items() if k not in ("config",)})

## 7. Visualize and inspect search results

In [ ]:
history_df = pd.read_csv(ROOT / "results" / "nas_search_history.csv")
display(history_df.sort_values("fitness", ascending=False).head(10))

plt.figure(figsize=(8, 4))
for gen, group in history_df.groupby("generation"):
    plt.scatter([gen] * len(group), group["val_accuracy"], label=f"Generation {gen}")
plt.xlabel("Generation"); plt.ylabel("Validation accuracy (%)")
plt.title("Evolutionary NAS: candidate validation accuracy")
plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(history_df["latency_ms_mean"], history_df["val_accuracy"],
            c=history_df["feasible"].map({True: 1, False: 0}))
plt.axvline(MAX_LATENCY_MS, linestyle="--", label="Latency constraint")
plt.xlabel("CPU latency (ms / image)"); plt.ylabel("Validation accuracy (%)")
plt.title("Accuracy vs measured CPU latency")
plt.legend(); plt.grid(True); plt.show()

## 8. Retrain the selected architecture from scratch

The best architecture is initialized with fresh random weights and trained on the complete CIFAR-10 training set. Training and test transforms are resized to the selected architecture's input resolution.

In [ ]:
best_config = best_result["config"]
final_model = SearchableCNN(best_config).to(DEVICE)

full_train_loader = make_resolution_loader(train_full, best_config["input_resolution"], shuffle=True)
final_test_loader = make_resolution_loader(test_set, best_config["input_resolution"], shuffle=False)

final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(final_optimizer, T_max=FINAL_EPOCHS)
final_history = []

for epoch in range(FINAL_EPOCHS):
    tr_loss, tr_acc = run_epoch(final_model, full_train_loader, final_optimizer)
    te_loss, te_acc = run_epoch(final_model, final_test_loader)
    scheduler.step()
    final_history.append({"epoch": epoch+1, "train_loss": tr_loss, "train_accuracy": tr_acc,
                          "test_loss": te_loss, "test_accuracy": te_acc})
    print(f"Epoch {epoch+1:02d}/{FINAL_EPOCHS} | train acc {tr_acc:.2f}% | test acc {te_acc:.2f}%")

final_checkpoint = ROOT / "checkpoints" / "best_nas_model.pth"
torch.save({
    "model_state_dict": final_model.state_dict(),
    "config": best_config,
    "class_names": train_full.classes,
    "test_accuracy": final_history[-1]["test_accuracy"],
}, final_checkpoint)
pd.DataFrame(final_history).to_csv(ROOT / "results" / "final_training_history.csv", index=False)
print("Saved checkpoint:", final_checkpoint)

## 9. Final CPU benchmark and deployment-constraint report

This benchmark is repeated on the final trained model. Latency values depend on the CPU, thread count, thermal state, and background load, so include the generated CSV and machine information in your report.

In [ ]:
final_hw = evaluate_hardware(final_model, best_config)
final_report = {
    "architecture": best_config,
    "test_accuracy": final_history[-1]["test_accuracy"],
    "hardware": final_hw,
    "constraints": {
        "max_latency_ms": MAX_LATENCY_MS,
        "max_model_size_mb": MAX_MODEL_SIZE_MB,
        "max_peak_memory_mb": MAX_PEAK_MEMORY_MB,
    },
    "machine": {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
        "torch_threads": torch.get_num_threads(),
    }
}
with open(ROOT / "results" / "final_benchmark.json", "w") as f:
    json.dump(final_report, f, indent=2)
pd.DataFrame([{
    **final_hw, "test_accuracy": final_history[-1]["test_accuracy"],
    "platform": platform.platform(), "cpu_count": os.cpu_count()
}]).to_csv(ROOT / "results" / "final_benchmark.csv", index=False)
print(json.dumps(final_report, indent=2))

## 10. Export selected model to ONNX

The exported graph has a dynamic batch dimension and a fixed spatial resolution matching the selected architecture. ONNX checker validates the graph before saving.

In [ ]:
onnx_path = ROOT / "onnx" / "best_nas_model.onnx"
final_model_cpu = copy.deepcopy(final_model).to("cpu").eval()
dummy_input = torch.randn(1, 3, best_config["input_resolution"], best_config["input_resolution"])

torch.onnx.export(
    final_model_cpu,
    dummy_input,
    str(onnx_path),
    input_names=["images"],
    output_names=["logits"],
    dynamic_axes={"images": {0: "batch_size"}, "logits": {0: "batch_size"}},
    opset_version=17,
    do_constant_folding=True,
)
if onnx is not None:
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
print("ONNX model exported:", onnx_path, "| size MB:", round(onnx_path.stat().st_size / (1024**2), 3))

## 11. ONNX Runtime CPU inference and parity check

ONNX Runtime is configured to use the CPU execution provider. The notebook compares PyTorch and ONNX outputs on the same input to verify that export preserved model behavior.

In [ ]:
if ort is None:
    raise ImportError("onnxruntime is not installed. Run: %pip install onnx onnxruntime")

session_options = ort.SessionOptions()
session_options.intra_op_num_threads = CPU_THREADS
session_options.inter_op_num_threads = 1
ort_session = ort.InferenceSession(str(onnx_path), sess_options=session_options,
                                   providers=["CPUExecutionProvider"])

test_batch, test_labels = next(iter(final_test_loader))
sample = test_batch[:8].cpu().numpy().astype(np.float32)
with torch.no_grad():
    pytorch_logits = final_model_cpu(torch.from_numpy(sample)).numpy()
onnx_logits = ort_session.run(["logits"], {"images": sample})[0]
max_abs_diff = float(np.max(np.abs(pytorch_logits - onnx_logits)))
print("ONNX Runtime providers:", ort_session.get_providers())
print("Maximum absolute logit difference:", max_abs_diff)
print("PyTorch predictions:", pytorch_logits.argmax(axis=1).tolist())
print("ONNX predictions:   ", onnx_logits.argmax(axis=1).tolist())

In [ ]:
def benchmark_onnx(session, input_resolution, warmup=30, iterations=200):
    x = np.random.randn(1, 3, input_resolution, input_resolution).astype(np.float32)
    input_name = session.get_inputs()[0].name
    for _ in range(warmup):
        session.run(None, {input_name: x})
    times = []
    for _ in range(iterations):
        start = time.perf_counter()
        session.run(None, {input_name: x})
        times.append((time.perf_counter() - start) * 1000.0)
    return {
        "onnx_latency_ms_mean": float(np.mean(times)),
        "onnx_latency_ms_p95": float(np.percentile(times, 95)),
    }

onnx_benchmark = benchmark_onnx(ort_session, best_config["input_resolution"])
onnx_size_mb = onnx_path.stat().st_size / (1024**2)
onnx_benchmark["onnx_model_size_mb"] = float(onnx_size_mb)
pd.DataFrame([onnx_benchmark]).to_csv(ROOT / "results" / "onnx_cpu_benchmark.csv", index=False)
print(onnx_benchmark)

## 12. Reusable inference function

Use this function to classify a CIFAR-10 image tensor with either the trained PyTorch model or the exported ONNX model.

In [ ]:
CLASS_NAMES = train_full.classes

def predict_pytorch(images_tensor):
    final_model_cpu.eval()
    with torch.no_grad():
        logits = final_model_cpu(images_tensor.cpu())
        probs = torch.softmax(logits, dim=1)
    confidence, indices = probs.max(dim=1)
    return [(CLASS_NAMES[i], float(c)) for i, c in zip(indices, confidence)]

def predict_onnx(images_numpy):
    outputs = ort_session.run(["logits"], {"images": images_numpy.astype(np.float32)})[0]
    probs = torch.softmax(torch.from_numpy(outputs), dim=1).numpy()
    indices = probs.argmax(axis=1)
    return [(CLASS_NAMES[i], float(probs[row, i])) for row, i in enumerate(indices)]

print("PyTorch sample predictions:", predict_pytorch(test_batch[:5]))
print("ONNX sample predictions:", predict_onnx(test_batch[:5].numpy()))

## 13. Save experiment summary and package deliverables



In [ ]:
summary = {
    "task": "Question 2 - Hardware-aware Neural Architecture Search",
    "search_strategy": "Evolutionary architecture search",
    "dataset": "CIFAR-10",
    "best_config": best_config,
    "final_test_accuracy": final_history[-1]["test_accuracy"],
    "pytorch_hardware_metrics": final_hw,
    "onnx_hardware_metrics": onnx_benchmark,
    "search_candidates_evaluated": len(search_records),
}
with open(ROOT / "results" / "experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Generated files:")
for p in sorted(ROOT.rglob("*")):
    if p.is_file():
        print(f"{p} ({p.stat().st_size/1024:.1f} KB)")